<a href="https://colab.research.google.com/github/BrandonDelM/animal_classification/blob/main/animalrecognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# import libraries
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
from sklearn.utils import class_weight

In [ ]:
# convert all image sizes to 64 x 64 pixels and divide by 255 to normalize images sizes betweehn [0, 255]
# [0, 255] helps model classify images to [0, 1] for easy classification
def preprocess(image, label):
    return tf.image.resize(image, (64, 64)) / 255.0, label # label: 0 for dog and 1 for cat

# load and split data from tensorflow libraries into train and test. Store as supervised data with image and its label
train_data, test_data = tfds.load("cats_vs_dogs", split=["train[:80%]", "train[20%:]"], as_supervised=True)

# preprocess train and test data into batches of 32 (of each label)
# put data in batches so model can learn faster by using parallel processing
# prefetch is data pipeline to streamline data being processed for next batch while creating current batch
train_data = train_data.map(preprocess).batch(32).prefetch(tf.data.AUTOTUNE)
test_data = test_data.map(preprocess).batch(32).prefetch(tf.data.AUTOTUNE)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...:   0%|          | 0/23262 [00:00<?, ? examples/s]

Shuffling /root/tensorflow_datasets/cats_vs_dogs/incomplete.5K5WLG_4.0.1/cats_vs_dogs-train.tfrecord*...:   0%…

Dataset cats_vs_dogs downloaded and prepared to /root/tensorflow_datasets/cats_vs_dogs/4.0.1. Subsequent calls will reuse this data.


In [ ]:
# Convolutional neural network (CNN) is a  neural network used for classification
# Sequential is an API in keras that stacks layers to build a CNN
model = tf.keras.Sequential([

    # The 16 filters (3 x 3) scan the input image, each detecting different features (ex: edges)
    # ReLU (Rectified Linear Unit) is an activation function that makes sure the layer focuses on important features
    # The model takes 64 x 64 pixel images with 3 color channels (RGB)
    tf.keras.layers.Conv2D(16, (3, 3), activation='relu', input_shape=(64, 64, 3)),

    # takes feature map created by Conv2D and shrinks it by using a 2 x 2 matrix and retains important information from each matrix
    tf.keras.layers.MaxPooling2D(2, 2),

    # looks for patterns/features from previous layer
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),

    # shrinks feature map made by previous layer and retains important info
    tf.keras.layers.MaxPooling2D(2, 2),

    # averages values for every channel (RGB) in map
    tf.keras.layers.GlobalAveragePooling2D(),

    # Classifier layer where 2 shows number of classes to indentify (cat or dog)
    # softmax is a activation function that converts the scores generated by previous layer into probability between [0, 1]
    tf.keras.layers.Dense(2, activation='softmax')
])

# configuring the model before training
# Optimizer (adam): makes training faster and reduces data loss
# Loss (sparse_categorical_crossentropy): measures how far the model’s predictions are from the actual values
# Metrics (accuracy): percentage of correct predictions made by the model during training
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
# collects all labels from the training dataset
labels = np.concatenate([y for _, y in train_data])

# assigns each label weights based on frequency (lower frequency == higher weight) so model focuses on all labels equally
class_weight_dict = dict(enumerate(class_weight.compute_class_weight('balanced', classes=np.unique(labels), y=labels)))

In [ ]:
# model will train on the train_data for 2 passes (epochs)
# validation_data=test_data: 20% test_data used to check if the model is learning not just memorizing (overfitting) images
# class_weight=class_weight_dict: model focuses on all classes equally
model.fit(train_data, validation_data=test_data, epochs=2, class_weight=class_weight_dict)

Epoch 1/2
582/582 ━━━━━━━━━━━━━━━━━━━━ 121s 204ms/step - accuracy: 0.5491 - loss: 0.6833 - val_accuracy: 0.6152 - val_loss: 0.6526
Epoch 2/2
582/582 ━━━━━━━━━━━━━━━━━━━━ 116s 199ms/step - accuracy: 0.6152 - loss: 0.6525 - val_accuracy: 0.6242 - val_loss: 0.6434


In [1]:
from tensorflow.keras.preprocessing import image

# load image of a cat or dog to test model
image_path = "/content/dog.jpg"

# resizes loaded image to 64 x 64 pixels
img = image.load_img(image_path, target_size=(64, 64))

# converts pixels to numbers and normalizes those numbers to be between [0, 1]
img_array = np.expand_dims(image.img_to_array(img), axis=0) / 255.0

# passes image into model
prediction = model.predict(img_array)

# returns the highest probability
label = np.argmax(prediction)

# return cat or dog based on prediction
if label == 0:
    print("It's a Dog!")
else:
    print("It's a Cat!")

KeyboardInterrupt: 